# Stress Test Example

janrth's original notebook is a **performance** stress test: it builds 50,000 synthetic
articles (all with the same deterministic demand pattern) and times
`optimize_aggregation_and_forecast_targets` picking a winning percentile target and
aggregation window across all of them, reporting only wall-clock time and the winning
target -- not a cost/fill-rate number.

**Scope note (Task 14 rewrite, same approach as Task 13's `test_golden_simulation.py`):**
`optimize_aggregation_and_forecast_targets`/`ForecastCandidatesConfig` (janrth's
`optimization.py`/`aggregation.py` ecosystem) were never ported into this repo -- see
`io_.py`'s module docstring, confirmed out of scope for Tasks 1-13. This notebook cannot
faithfully replay the original optimizer call, so it demonstrates the same *kind* of
thing -- many articles, each getting a `ReplenishmentPolicy` built per percentile
candidate via `NullSafetyStockStrategy`, run through `simulate_replenishment`, timed --
using only what this repo actually has. **Article count is reduced from 50,000 to 500**
to keep demo runtime reasonable; the demand pattern, cost parameters, and percentile
candidate grid are otherwise identical to the original notebook. There is no aggregation
window sweep here (that ecosystem is unported), so this measures per-article percentile
selection throughput only, not the original's window+target joint search.

In [1]:
import time
from collections import Counter

from replenishment.simulation import simulate_replenishment
from replenishment.policy import ReplenishmentPolicy
from replenishment.timeseries import TimeSeries
from replenishment.strategies.multiplier import NullSafetyStockStrategy

### Policy cadence parameters

In [2]:
review_period = 1
forecast_horizon = 1

### Scenario parameters (same as janrth's original, article_count reduced)

In [3]:
article_count = 500  # janrth's original used 50_000
periods = 120
lead_time = 1
holding_cost_per_unit = 0.8
stockout_cost_per_unit = 3.5
initial_on_hand = 40

base_demand = [50 + (idx % 10) for idx in range(periods)]
percentile_targets = list(range(10, 85, 5))

def _percentile_offset(target: int) -> int:
    return int(round((target - 50) / 5))

forecast_candidates = {
    f"p{target}": [value + _percentile_offset(target) for value in base_demand]
    for target in percentile_targets
}

### Build one `ReplenishmentPolicy` per candidate target, per article, and time the selection

In [4]:
def best_target_for_article(demand):
    best_target = None
    best_cost = None
    for target, series in forecast_candidates.items():
        policy = ReplenishmentPolicy.order_up_to(
            forecast=TimeSeries.from_values(series),
            safety_stock=NullSafetyStockStrategy(),
            lead_time=lead_time, review_period=review_period, forecast_horizon=forecast_horizon,
        )
        simulation = simulate_replenishment(
            periods=periods, demand=demand, initial_on_hand=initial_on_hand, lead_time=lead_time,
            policy=policy, holding_cost_per_unit=holding_cost_per_unit,
            stockout_cost_per_unit=stockout_cost_per_unit,
        )
        cost = simulation.summary.total_cost
        if best_cost is None or cost < best_cost:
            best_cost = cost
            best_target = target
    return best_target, best_cost


start = time.perf_counter()
best_targets = {}
for idx in range(article_count):
    article_id = f"article-{idx:05d}"
    best_targets[article_id] = best_target_for_article(base_demand)
elapsed = time.perf_counter() - start

print(f"Total time: {elapsed:.2f}s")
print(f"Time per article: {elapsed / article_count:.6f}s")

target_counts = Counter(target for target, _ in best_targets.values())
print(dict(target_counts))

Total time: 3.82s
Time per article: 0.007645s
{'p50': 500}


In [5]:
best_targets["article-00000"]

('p50', 35.0)